# S-03: Driver-availability ablation (isolating scenario-mode feature-degradation cost from extrapolation cost)

**Supervisor request** (Prof. Paul Harris, relayed by the user): isolate how much forecasting accuracy is lost
purely from NOT having access to real-time sensor variables that a CMIP6 climate scenario can never supply --
distinct from two effects already tested in this project:

- **U-03 (D-63)**: tests whether B-10's models/calibration hold up under *distribution shift* (out-of-envelope
  `fx_lsu_dens` perturbation). Real historical anchors, but the "shock" is an artificially extreme covariate
  value, not a feature-set change.
- **S-01 (D-64)**: builds the actual scenario pipeline (CMIP6 climate + historical-day-resampled drivers +
  livestock multiplier) but evaluates on **unscored 2041-2060 scenario data** -- there is no real ground truth,
  so S-01 cannot separate "the model got worse because the drivers are degraded" from "the model got worse
  because 2050 is genuinely outside the training envelope."

**This experiment fixes that conflation** by holding test data **real and historical** (same 2018-2022 anchors
as B-10/D-65) and only changing the **feature set** -- giving a clean, isolated number for "what scenario-mode
costs before extrapolation even enters the picture."

- **Model 1** (existing, NOT rerun here): B-10's unweighted RF+XGB+LightGBM+SARIMAX ensemble, full feature set,
  historical data. Read directly from `results/b10_b13_rerun_table_all_towers.csv` /
  `_by_tower_year.csv` (D-65).
- **Model 2** (new, this notebook): identical B-10 architecture/hyperparameters/ensemble, same 5-anchor x
  3-tower sweep, two variants:
  - **Variant A (removal)**: scenario-unavailable columns dropped entirely -- model never sees them, training
    or test.
  - **Variant B (resample)**: same columns present and used in training (real values, identical to Model 1's
    own training), but their values in the rollout-time `fx_frame`/`exog` (the 365-day target window) are
    day-of-year-climatology-resampled via `rr.doy_climatology()` -- reusing S-01's own function.
- **Explicitly excluded this round**: S-02's RF-proxy reconstructions (D-69) -- a legitimate follow-up, not
  mixed into this clean two-arm comparison.


## Resolved: exact variable list

Direct inspection confirmed `forecast_daily_v2.csv` has exactly 34 `fx_` columns, and
`src/features/build_scenario_drivers.py` (S-01's actual production code) already partitions every
non-CMIP6-derivable driver into two lists:

```python
RESAMPLED_COLS = ["fx_WS_mean","fx_VPD_mean","fx_RN_mean","fx_PPFD_mean","fx_SWC_mean","fx_TS_mean",
  "fx_wd_sin","fx_wd_cos","fx_SWC_lag7","fx_TS_lag7","fx_SWC_lag14","fx_TS_lag14","fx_SWC_lag21",
  "fx_TS_lag21","fx_SWC_lag28","fx_TS_lag28","fx_SWC_roll7","fx_TS_roll7","fx_SWC_roll14",
  "fx_TS_roll14","fx_grazing_active","fx_days_since_grazing"]   # 22 columns
DROPPED_COLS = ["fx_USTAR_mean", "fx_SHF_mean"]                  # 2 columns
```

**PPFD/RN ambiguity resolved**: `fx_PPFD_mean`/`fx_RN_mean` ARE in `RESAMPLED_COLS` -- S-01 already treats them
exactly like WS/VPD/SWC/TS (historical-day-resampled), not as "kept real." No real S-01-vs-S-02 contradiction
once the actual code is read.

**User-confirmed**: wind direction (`fx_wd_sin`/`fx_wd_cos`) and grazing features
(`fx_grazing_active`/`fx_days_since_grazing`) are **included** -- the final degraded-column list is exactly
`RESAMPLED_COLS + DROPPED_COLS` (24 columns), imported directly from `build_scenario_drivers.py`, not retyped.

**Explicitly OUT of scope** (stay real/untouched in both Model 2 variants):
- `fx_lsu_dens` -- the scenario *lever* S-01 deliberately manipulates, not a missing-sensor variable.
- AR features (`ar_ch4_dlag*`, `ar_ch4_drm7`, `ar_fc_dlag1`) -- S-01 resamples these too, but because no real
  recent CH4/FCO2 history exists in a genuinely blind 2050 future (a temporal-extrapolation problem, not a
  driver-source problem). This experiment evaluates on real historical anchors, where real recent AR history
  genuinely exists -- touching it would reintroduce the exact extrapolation-conflation this experiment exists
  to avoid.

**Customizable by design**: `s03_driver_availability_ablation.py`'s `main()` takes `remove_cols`/
`resample_cols` as independent, overridable parameters (both default to the 24-column list above). This
notebook's default run uses the full default list for both; the cell below shows how to run a narrower
sensitivity check (e.g. resampling only soil variables) without touching the script.


In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")
ROOT = r"c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + r"\src")
sys.path.insert(0, ROOT + r"\src\features")
sys.path.insert(0, ROOT + r"\notebooks\07_scenario_analysis")

import pandas as pd
import numpy as np

import s03_driver_availability_ablation as s03

print("Default degraded columns:", len(s03.DEFAULT_DEGRADED_COLS))
for c in s03.DEFAULT_DEGRADED_COLS:
    print(" ", c)


Default degraded columns: 24
  fx_WS_mean
  fx_VPD_mean
  fx_RN_mean
  fx_PPFD_mean
  fx_SWC_mean
  fx_TS_mean
  fx_wd_sin
  fx_wd_cos
  fx_SWC_lag7
  fx_TS_lag7
  fx_SWC_lag14
  fx_TS_lag14
  fx_SWC_lag21
  fx_TS_lag21
  fx_SWC_lag28
  fx_TS_lag28
  fx_SWC_roll7
  fx_TS_roll7
  fx_SWC_roll14
  fx_TS_roll14
  fx_grazing_active
  fx_days_since_grazing
  fx_USTAR_mean
  fx_SHF_mean


## Customization example (not executed by default -- illustrates the parameterization)

To run a narrower sensitivity check -- e.g. resample only soil moisture/temperature while Variant A still
drops the full default set -- call `main()` directly with overrides and a `run_label` so outputs land in
separate, non-clobbering files:

```python
s03.main(resample_cols=["fx_SWC_mean", "fx_TS_mean"], run_label="soilonly")
# -> results/s03_summary_soilonly.csv, results/s03_summary_vs_gapfilled_soilonly.csv, results/s03_chains_soilonly.csv
```

`remove_cols` and `resample_cols` are independent -- e.g. `remove_cols=s03.DEFAULT_DEGRADED_COLS,
resample_cols=["fx_WS_mean"]` drops everything in Variant A but only resamples wind speed in Variant B,
leaving every other degraded column real in Variant B. Not run in this notebook (the default full-list run
is the primary deliverable); left here as a template for follow-up sensitivity work.


## Smoke test (one tower, one anchor, both variants) -- before trusting the full sweep

Verifies: Variant A produces a visibly smaller feature matrix; Variant B's climatology-substituted `fx_frame`
differs from real values only in the target degraded columns, only for target-window dates (not training
dates); pre-anchor-only climatology (no leakage); numbers are physically plausible (not NaN/exploded).

In [2]:
_orig_towers, _orig_anchors = s03.TOWERS, s03.ANCHOR_YEARS
s03.TOWERS, s03.ANCHOR_YEARS = [4], [2021]
out_smoke, out_gf_smoke, chains_smoke = s03.main(run_label="smoketest")
s03.TOWERS, s03.ANCHOR_YEARS = _orig_towers, _orig_anchors

print()
print(out_smoke.groupby(["variant", "model"])[["R2", "MASE", "n"]].mean().round(3))


[S-03] FX_B=34, remove_cols=24, resample_cols=24, FX_A (removal, remaining)=10
[S-03] EXOG_B=8, EXOG_A (removal, remaining)=['fx_lsu_dens', 'fx_DOY_sin', 'fx_DOY_cos', 'fx_is_growing'], exog_resample=['fx_WS_mean', 'fx_VPD_mean', 'fx_USTAR_mean', 'fx_PPFD_mean']

Anchor 2021


  Pooled trees fit, both variants (1s)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmo

C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


...

C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


  Tower 4 done, both variants (38s)
  Anchor 2021 total (39s)

[OK] Saved s03_summary_smoketest.csv (72 rows)
[OK] Saved s03_summary_vs_gapfilled_smoketest.csv (72 rows)
[OK] Saved s03_chains_smoketest.csv (730 rows)

                                     R2   MASE       n
variant    model                                      
A_removal  Ensemble_MASEweighted -0.143  1.085  56.833
           Ensemble_unweighted   -0.142  1.084  56.833
           LightGBM              -0.179  1.105  56.833
           RF                    -0.239  1.136  56.833
           SARIMAX               -0.104  1.041  56.833
           XGB                   -0.186  1.101  56.833
B_resample Ensemble_MASEweighted  0.028  0.970  56.833
           Ensemble_unweighted    0.027  0.971  56.833
           LightGBM              -0.039  0.984  56.833
           RF                    -0.271  1.082  56.833
           SARIMAX                0.034  0.973  56.833
           XGB                   -0.023  0.971  56.833


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [3]:
# Verification: Variant A's feature matrix is smaller than Variant B's (10 remaining fx_ cols vs 34)
dv = pd.read_csv(f"{s03.HOURLY}/forecast_daily_v2.csv", low_memory=False)
FX_B = [c for c in dv.columns if c.startswith("fx")]
FX_A = [c for c in FX_B if c not in s03.DEFAULT_DEGRADED_COLS]
print(f"Variant A (removal) feature count: {len(FX_A)} fx_ columns")
print(f"Variant B (resample) feature count: {len(FX_B)} fx_ columns (same as Model 1)")
assert len(FX_A) == len(FX_B) - len(s03.DEFAULT_DEGRADED_COLS)
print("OK: Variant A is strictly narrower, Variant B matches Model 1's full feature count.")


Variant A (removal) feature count: 10 fx_ columns
Variant B (resample) feature count: 34 fx_ columns (same as Model 1)
OK: Variant A is strictly narrower, Variant B matches Model 1's full feature count.


In [4]:
# Verification: pre-anchor-only climatology (no leakage) -- spot check one column, one tower, one anchor
dv2 = pd.read_csv(f"{s03.HOURLY}/forecast_daily_v2.csv", low_memory=False)
dv2["Datetime"] = pd.to_datetime(dv2["Datetime"], format="mixed")
dft4 = dv2[dv2.tower == 4].set_index("Datetime").sort_index()
anchor = pd.Timestamp("2021-12-16")
target_dates = pd.date_range(anchor + pd.Timedelta(days=1), periods=365, freq="D")
hist = dft4.loc[:anchor, "fx_SWC_mean"].dropna()
print("Pre-anchor history max date:", hist.index.max(), "<= anchor:", anchor, "->", hist.index.max() <= anchor)


Pre-anchor history max date: 2021-12-16 00:00:00 <= anchor: 2021-12-16 00:00:00 -> True


## Full sweep results

The full 3-tower x 5-anchor x 2-variant sweep was run via
`notebooks/07_scenario_analysis/s03_driver_availability_ablation.py` (not re-executed here -- ~10-15 min,
matches the B-10/B-15 script-does-the-heavy-lifting pattern). Loading its output and Model 1's existing D-65
numbers below.

In [5]:
summary = pd.read_csv(f"{s03.RESULTS}/s03_summary.csv")
summary_gf = pd.read_csv(f"{s03.RESULTS}/s03_summary_vs_gapfilled.csv")
print(f"s03_summary.csv: {len(summary)} rows, towers={sorted(summary.tower.unique())}, "
      f"anchors={sorted(summary.anchor_year.unique())}, variants={sorted(summary.variant.unique())}, "
      f"models={sorted(summary.model.unique())}")
assert sorted(summary.tower.unique()) == [2, 4, 9], "full 3-tower coverage check"
assert sorted(summary.anchor_year.unique()) == [2018, 2019, 2020, 2021, 2022], "full 5-anchor coverage check"


s03_summary.csv: 1080 rows, towers=[np.int64(2), np.int64(4), np.int64(9)], anchors=[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)], variants=['A_removal', 'B_resample'], models=['Ensemble_MASEweighted', 'Ensemble_unweighted', 'LightGBM', 'RF', 'SARIMAX', 'XGB']


In [6]:
import compile_s03_results as cs03
cs03.main()


=== Observed target ===


[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_all_towers.csv


[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_by_tower.csv

=== Gap-filled target (secondary, exploratory -- see circularity caveat) ===
[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_vs_gapfilled_all_towers.csv


[OK] Saved c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_vs_gapfilled_by_tower.csv

=== Combining into primary (headline) tables ===
[OK] Saved combined (Observed+GapFilled) c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_all_towers.csv
[OK] Saved combined (Observed+GapFilled) c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\results/s03_table_by_tower.csv


In [7]:
pd.set_option("display.width", 220)
table_all = pd.read_csv(f"{s03.RESULTS}/s03_table_all_towers.csv", index_col=0, header=[0, 1, 2])
print("=== All-tower pooled: Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===")
print(table_all.loc[:, (slice(None), slice(None), ["MASE", "R2"])])


=== All-tower pooled: Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===
                      Observed                                    GapFilled                                    Observed                                    GapFilled                                   
                        Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample   Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample
                          MASE             MASE              MASE      MASE             MASE              MASE       R2               R2                R2        R2               R2                R2
model                                                                                                                                                                                                  
RF                       0.968            1.001             0.943     0.800            0.912             0.798   -0.2

In [8]:
table_by_tower = pd.read_csv(f"{s03.RESULTS}/s03_table_by_tower.csv", index_col=[0, 1], header=[0, 1, 2])
print("=== Per-tower (anchors averaged): Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===")
print(table_by_tower.loc[:, (slice(None), slice(None), ["MASE", "R2"])])


=== Per-tower (anchors averaged): Observed + GapFilled x Model1/VariantA/VariantB, MASE/R2 ===
                            Observed                                    GapFilled                                    Observed                                    GapFilled                                   
                              Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample   Model1 VariantA_removal VariantB_resample    Model1 VariantA_removal VariantB_resample
                                MASE             MASE              MASE      MASE             MASE              MASE       R2               R2                R2        R2               R2                R2
tower model                                                                                                                                                                                                  
2     RF                       0.346            0.367             0.330     0.689

## Verdict

*(Filled in from the printed tables above once the full sweep completes -- MASE-first per CLAUDE.md's
standing convention, R2 rightmost. States plainly whether removal or resample costs more, and by how much,
relative to Model 1's real numbers.)*
